### basic-streaming(node-level)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
import time

class State(TypedDict):
    topic: str
    joke: str

def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}

def generate_joke(state: State):
    time.sleep(1)
    return {"joke": f"This is a joke about {state['topic']}"}

graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .add_edge("generate_joke", END)
    .compile()
)
for chunk in graph.stream(  
    {"topic": "ice cream"},
    stream_mode="updates",  
):
    print(chunk)

{'refine_topic': {'topic': 'ice cream and cats'}}
{'generate_joke': {'joke': 'This is a joke about ice cream and cats'}}


With `stream_mode="updates"`, it returns **only the output of each node as it runs**, not the full state.



### Multiple modes

In [17]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import StreamWriter

class State(TypedDict):
    topic: str
    joke: str

def refine_topic(state: State, writer: StreamWriter):
    writer({"stage": "refining topic"})
    
    updated_topic = state["topic"] + " and cats"
    return {"topic": updated_topic}


def generate_joke(state: State, writer: StreamWriter):
    writer({"stage": "generating joke"})
    
    joke = f"This is a joke about {state['topic']}"
    return {"joke": joke}

builder = StateGraph(State)

builder.add_node("refine_topic", refine_topic)
builder.add_node("generate_joke", generate_joke)

builder.add_edge(START, "refine_topic")
builder.add_edge("refine_topic", "generate_joke")
builder.add_edge("generate_joke", END)

graph = builder.compile()

inputs = {"topic": "ice cream"}

for mode, chunk in graph.stream(
    inputs,
    stream_mode=["updates", "custom","debug"] 
):
    print(f"\nMode: {mode}")
    print("Chunk:", chunk)


Mode: debug
Chunk: {'step': 1, 'timestamp': '2026-02-25T11:15:25.281205+00:00', 'type': 'task', 'payload': {'id': 'ae2ed52a-94d1-17f2-10a4-3afe4d59aa07', 'name': 'refine_topic', 'input': {'topic': 'ice cream'}, 'triggers': ('branch:to:refine_topic',)}}

Mode: custom
Chunk: {'stage': 'refining topic'}

Mode: updates
Chunk: {'refine_topic': {'topic': 'ice cream and cats'}}

Mode: debug
Chunk: {'step': 1, 'timestamp': '2026-02-25T11:15:25.282270+00:00', 'type': 'task_result', 'payload': {'id': 'ae2ed52a-94d1-17f2-10a4-3afe4d59aa07', 'name': 'refine_topic', 'error': None, 'result': {'topic': 'ice cream and cats'}, 'interrupts': []}}

Mode: debug
Chunk: {'step': 2, 'timestamp': '2026-02-25T11:15:25.282765+00:00', 'type': 'task', 'payload': {'id': 'c8aa3d9a-8b94-0a66-692d-b11c832b1e1c', 'name': 'generate_joke', 'input': {'topic': 'ice cream and cats'}, 'triggers': ('branch:to:generate_joke',)}}

Mode: custom
Chunk: {'stage': 'generating joke'}

Mode: updates
Chunk: {'generate_joke': {'joke'

* `"updates"` → streams node return values.
* `"custom"` → streams data sent manually via `StreamWriter`.
* `"debug"` → streams detailed execution info for tracing.




### Full-state streaming (modes=values)

In [16]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
  topic: str
  joke: str


def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}


def generate_joke(state: State):
    return {"joke": f"This is a joke about {state['topic']}"}

graph = (
  StateGraph(State)
  .add_node(refine_topic)
  .add_node(generate_joke)
  .add_edge(START, "refine_topic")
  .add_edge("refine_topic", "generate_joke")
  .add_edge("generate_joke", END)
  .compile()
)

for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="values",  
):
    print(chunk)

{'topic': 'ice cream'}
{'topic': 'ice cream and cats'}
{'topic': 'ice cream and cats', 'joke': 'This is a joke about ice cream and cats'}


With `stream_mode="values"`, the stream returns the **entire state after each step**, not just the updates.

So after each node runs, you receive the full current state (`topic`, then `topic + joke`).


### Subgraph streaming

In [ ]:
from langgraph.graph import START, StateGraph
from typing import TypedDict

# Define subgraph
class SubgraphState(TypedDict):
    foo: str  # note that this key is shared with the parent graph state
    bar: str

def subgraph_node_1(state: SubgraphState):
    return {"bar": "bar"}

def subgraph_node_2(state: SubgraphState):
    return {"foo": state["foo"] + state["bar"]}

subgraph_builder = StateGraph(SubgraphState)
subgraph_builder.add_node(subgraph_node_1)
subgraph_builder.add_node(subgraph_node_2)
subgraph_builder.add_edge(START, "subgraph_node_1")
subgraph_builder.add_edge("subgraph_node_1", "subgraph_node_2")
subgraph = subgraph_builder.compile()

# Define parent graph
class ParentState(TypedDict):
    foo: str

def node_1(state: ParentState):
    return {"foo": "hi! " + state["foo"]}

builder = StateGraph(ParentState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", subgraph)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
graph = builder.compile()

for chunk in graph.stream(
    {"foo": "foo"},
    stream_mode="updates",
    # Set subgraphs=True to stream outputs from subgraphs
    subgraphs=True,  
):
    print(chunk)

((), {'node_1': {'foo': 'hi! foo'}})
(('node_2:001136ab-6855-0f04-27be-3a30890ebbba',), {'subgraph_node_1': {'bar': 'bar'}})
(('node_2:001136ab-6855-0f04-27be-3a30890ebbba',), {'subgraph_node_2': {'foo': 'hi! foobar'}})
((), {'node_2': {'foo': 'hi! foobar'}})


With `subgraphs=True` and `stream_mode="updates"`, it also streams the updates from inside the subgraph nodes, not just the parent nodes.


In [18]:
from dotenv import load_dotenv
load_dotenv()

True

### LLM Token Streaming

In [21]:
from dataclasses import dataclass

from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START


@dataclass
class MyState:
    topic: str
    explaination: str = ""


model = init_chat_model(model="google_genai:gemini-2.5-flash")

def call_model(state: MyState):
    """Call the LLM to generate a explaination about a topic"""
    # Note that message events are emitted even when the LLM is run using .invoke rather than .stream
    model_response = model.invoke(  
        [
            {"role": "user", "content": f"Generate a explaination about {state.topic}"}
        ]
    )
    return {"explaination": model_response.content}

graph = (
    StateGraph(MyState)
    .add_node(call_model)
    .add_edge(START, "call_model")
    .compile()
)

# The "messages" stream mode returns an iterator of tuples (message_chunk, metadata)
# where message_chunk is the token streamed by the LLM and metadata is a dictionary
# with information about the graph node where the LLM was called and other information
for message_chunk, metadata in graph.stream(
    {"topic": "house of dragon"},
    stream_mode="messages",  
):
    if message_chunk.content:
        print(message_chunk.content, end="|", flush=True)

**House of the Dragon** is an American fantasy drama television series that serves as a **prequel** to the immensely popular *Game of Thrones*.|

Here's a breakdown of what it's all about:

1.  **Based On:** It's based on parts of George R.R. Martin's novel **"Fire & Blood,"** which chronicles the history| of House Targaryen.

2.  **Setting & Time:** The show is set approximately **200 years before the events of *Game of Thrones***, during the height of House Targaryen's power in Wester|os, when they possessed numerous dragons and ruled the Seven Kingdoms.

3.  **Core Focus: House Targaryen:** The series primarily focuses on **House Targaryen**, the infamous dragon-riding dynasty. It delves into their internal politics|, ambitions, and the intricate web of family relationships that ultimately lead to their downfall.

4.  **The Central Conflict: The Dance of the Dragons:** The main storyline revolves around the events leading up to, and eventually, the brutal Targ|aryen civil war known 

With `stream_mode="messages"`, it yields `(message_chunk, metadata)` tuples:

* `message_chunk` → token/content piece from the LLM
* `metadata` → info about which node triggered the LLM call


### Tagged LLM Streaming (Filter by Tags)

In [ ]:
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langgraph.graph import START, StateGraph

# The joke_model is tagged with "joke"
joke_model = init_chat_model(model="google_genai:gemini-2.5-flash", tags=["joke"])
# The poem_model is tagged with "poem"
poem_model = init_chat_model(model="google_genai:gemini-2.5-flash", tags=["poem"])


class State(TypedDict):
      topic: str
      joke: str
      poem: str


async def call_model(state, config):
      topic = state["topic"]
      print("Writing joke...")
      
      joke_response = await joke_model.ainvoke(
            [{"role": "user", "content": f"Write a joke about {topic}"}],
            config,
      )
      print("\n\nWriting poem...")
      poem_response = await poem_model.ainvoke(
            [{"role": "user", "content": f"Write a short poem about {topic}"}],
            config,
      )
      return {"joke": joke_response.content, "poem": poem_response.content}


graph = (
      StateGraph(State)
      .add_node(call_model)
      .add_edge(START, "call_model")
      .compile()
)

async for msg, metadata in graph.astream(
      {"topic": "cats"},
      stream_mode="messages",
):
    if metadata["tags"] == ["joke"]:
        print(msg.content, end="|", flush=True)

Writing joke...
Here's one for you:

Why did the cat knock all the pens off the table?

To| see if you were paying attention!||

Writing poem...


With `stream_mode="messages"`, each streamed token includes metadata (like `tags`).

By checking `metadata["tags"]`, you can filter and stream output from a specific model (e.g., only `"joke"`), even when multiple LLMs are used.


### Parallel Node LLM Streaming

In [ ]:
from typing import TypedDict
from langgraph.graph import START, StateGraph
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


class State(TypedDict):
      topic: str
      joke: str
      poem: str


def write_joke(state: State):
      topic = state["topic"]
      joke_response = model.invoke(
            [{"role": "user", "content": f"Write a joke about {topic}"}]
      )
      return {"joke": joke_response.content}


def write_poem(state: State):
      topic = state["topic"]
      poem_response = model.invoke(
            [{"role": "user", "content": f"Write a short poem about {topic}"}]
      )
      return {"poem": poem_response.content}


graph = (
      StateGraph(State)
      .add_node(write_joke)
      .add_node(write_poem)
      # write both the joke and the poem concurrently
      .add_edge(START, "write_joke")
      .add_edge(START, "write_poem")
      .compile()
)

for msg, metadata in graph.stream(
    {"topic": "cats"},
    stream_mode="messages",  
):
    if msg.content and metadata["langgraph_node"] == "write_poem":
        print(msg.content, end="|", flush=True)

Why did the cat sit on| the computer keyboard?

Because it wanted to keep an eye on the *mouse*!|

With `stream_mode="messages"`, it streams LLM tokens from both nodes as they generate.

By checking `metadata["langgraph_node"]`, you can filter and display output from a specific node (e.g., only `"write_poem"`).


### Custom Streaming with get_stream_writer()

In [25]:
from typing import TypedDict
from langgraph.config import get_stream_writer
from langgraph.graph import StateGraph, START

class State(TypedDict):
    query: str
    answer: str

def node(state: State):
    # Get the stream writer to send custom data
    writer = get_stream_writer()
    # Emit a custom key-value pair (e.g., progress update)
    writer({"custom_key": "Generating custom data inside node"})
    return {"answer": "some data"}

graph = (
    StateGraph(State)
    .add_node(node)
    .add_edge(START, "node")
    .compile()
)

inputs = {"query": "example"}

# Set stream_mode="custom" to receive the custom data in the stream
for chunk in graph.stream(inputs, stream_mode="custom"):
    print(chunk)

{'custom_key': 'Generating custom data inside node'}


Using `get_stream_writer()`, the node sends manual updates during execution.

With `stream_mode="custom"`, the stream returns only the custom data written via the writer, not the node’s returned state.


### Tool Progress + State Streaming

In [35]:
from typing import TypedDict
from langchain.tools import tool
from langgraph.graph import StateGraph, START
from langgraph.config import get_stream_writer


class State(TypedDict):
    query: str
    answer: str


@tool
def query_database_tool(query: str) -> str:
    """demooooo"""
    writer = get_stream_writer()
    writer({"data": "Retrieved 0/100 records", "type": "progress"})
    writer({"data": "Retrieved 100/100 records", "type": "progress"})
    return "some-answer"


def query_database_node(state: State):
    result = query_database_tool.invoke(state["query"])
    return {"answer": result}  # ✅ MUST return dict


graph = (
    StateGraph(State)
    .add_node("query_database", query_database_node)
    .add_edge(START, "query_database")
    .compile()
)

inputs = {"query": "how to fix this"}

for mode, chunk in graph.stream(inputs, stream_mode=["custom", "updates"]):
    print(mode, chunk)

custom {'data': 'Retrieved 0/100 records', 'type': 'progress'}
custom {'data': 'Retrieved 100/100 records', 'type': 'progress'}
updates {'query_database': {'answer': 'some-answer'}}


* The tool uses `get_stream_writer()` to emit progress messages.
* `stream_mode="custom"` → streams tool progress data.
* `stream_mode="updates"` → streams the node’s returned state (`answer`).

It allows real-time tool progress tracking while still updating graph state.
